# Deep baseline
* split from refac/01_data_cleansing.ipynb, stratified by method x charge, z in {2,3,4})
* full train split, early stopping on val split (patience 5, max 50 epochs), best checkpoint kept
* results in `results/runs.jsonl`


## Config: edit paths here

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
BRANCH = "day1-baseline"
REPO_URL = "https://github.com/LordBurtz/sommer.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/sommer"          # clean_spectra_split.parquet
    SAVE_DIR = "/content/drive/MyDrive/sommer/day1"     # results/checkpoints copied here at the end
    if not os.path.exists("/content/sommer"):
        !git clone -b {BRANCH} {REPO_URL} /content/sommer
    os.chdir("/content/sommer")
else:
    DATA_DIR = "refac"
    SAVE_DIR = None

DATA_FILE = "clean_spectra_split.parquet"
OUT_DIR = "."
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd(), "| data:", DATA_DIR)

In [ ]:
import train_deep

common_args = ["--data-dir", DATA_DIR, "--data-file", DATA_FILE, "--out-dir", OUT_DIR]
_ = train_deep.main(common_args + ["--smoke"])

Full training run


In [ ]:
record = train_deep.main(common_args)

## Results

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

runs = [json.loads(l) for l in open(os.path.join(OUT_DIR, "results/runs.jsonl"))]
print(pd.DataFrame([{
    "timestamp": r["timestamp"], "commit": r["git_commit"][:8], "device": r["device"],
    "n_train": r["n_train"], "epochs": r["epochs_trained"], "best_epoch": r["best_epoch"],
    "val_SA": r["val_spectral_angle"], "val_pearson": r["val_pearson"],
    **{f"SA_z{z}": v["spectral_angle"] for z, v in r["by_charge"].items()},
} for r in runs]).round(4).to_string(index=False))
display(Image(os.path.join(OUT_DIR, "results/deep_baseline_curve.png")))

In [ ]:
if SAVE_DIR:
    import shutil
    os.makedirs(SAVE_DIR, exist_ok=True)
    for d in ("results", "checkpoints"):
        shutil.copytree(os.path.join(OUT_DIR, d), os.path.join(SAVE_DIR, d), dirs_exist_ok=True)
    print("copied results/ and checkpoints/ to", SAVE_DIR)